# 🦾 Titan Browser — Colab Runner
| # | Cellule | Action |
|---|---------|--------|
| 1 | Config | PAT + variables |
| 2 | Ressources | CPU / RAM / Disk |
| 3 | Install | Télécharge le runner |
| 4 | Start | Enregistre + démarre |

> 💡 Sauvegarde ton PAT dans **Tools → Secrets → `GITHUB_PAT`** pour ne plus le retaper.
> ⚠️ Ne ferme pas l'onglet pendant le build.

In [ ]:
# Cellule 1 — Config
import os, getpass

REPO          = "ferelking242/titan-browser"
RUNNER_NAME   = "colab-runner"
RUNNER_LABELS = "self-hosted,colab,linux,x64"

os.environ['RUNNER_ALLOW_RUNASROOT'] = '1'

GITHUB_PAT = ""
try:
    from google.colab import userdata
    GITHUB_PAT = userdata.get('GITHUB_PAT') or ""
    if GITHUB_PAT:
        print("✅ PAT chargé depuis Secrets Colab")
except Exception:
    pass

if not GITHUB_PAT:
    GITHUB_PAT = getpass.getpass("🔑 GitHub PAT (repo + workflow) : ")

if not GITHUB_PAT:
    raise ValueError("❌ PAT vide — relance la cellule")

os.environ['GITHUB_PAT'] = GITHUB_PAT
print(f"✅ Config OK  |  {REPO}  |  runner: {RUNNER_NAME}")

In [ ]:
# Cellule 2 — Ressources
import subprocess, shutil

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

cpu_cores = sh('nproc')
cpu_model = sh("lscpu | grep 'Model name' | awk -F':' '{print $2}'").strip()
ram_total = sh("free -h | awk '/Mem:/{print $2}'")
_, _, free = shutil.disk_usage("/")
free_gb = free / (1024**3)
disk_status = "✅" if free_gb >= 60 else "⚠️"
disk_note   = "" if free_gb >= 60 else " — besoin ~80 GB → Runtime → Disconnect and delete runtime"

print(f"CPU  : {cpu_cores} cores — {cpu_model}")
print(f"RAM  : {ram_total}")
print(f"Disk : {disk_status} {free_gb:.0f} GB libres{disk_note}")

In [ ]:
# Cellule 3 — Installation du runner
import subprocess, os, urllib.request

RUNNER_VERSION = "2.325.0"
RUNNER_DIR     = "/root/actions-runner"
runner_tar     = "/tmp/actions-runner.tar.gz"
runner_url     = ("https://github.com/actions/runner/releases/download"
                  f"/v{RUNNER_VERSION}/actions-runner-linux-x64-{RUNNER_VERSION}.tar.gz")

os.makedirs(RUNNER_DIR, exist_ok=True)

print(f"⬇️  Téléchargement runner v{RUNNER_VERSION}...")
urllib.request.urlretrieve(runner_url, runner_tar)

print("📦 Extraction...")
subprocess.run(["tar", "xzf", runner_tar, "-C", RUNNER_DIR], check=True)

print("🔧 Dépendances...")
r = subprocess.run(
    f"cd {RUNNER_DIR} && bash bin/installdependencies.sh",
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print("⚠️  stderr:", r.stderr[-500:])

print("✅ Runner prêt")

In [ ]:
# Cellule 4 — Enregistrement + démarrage
import requests, subprocess

RUNNER_DIR = "/root/actions-runner"

print("🔑 Obtention du token d'enregistrement...")
resp = requests.post(
    f"https://api.github.com/repos/{REPO}/actions/runners/registration-token",
    headers={"Authorization": f"token {GITHUB_PAT}", "Accept": "application/vnd.github.v3+json"}
)
if resp.status_code != 201:
    raise RuntimeError(f"❌ API GitHub {resp.status_code}: {resp.text}")
reg_token = resp.json()["token"]
print(f"✅ Token OK (expire: {resp.json()['expires_at']})")

print("📝 Enregistrement...")
reg = subprocess.run(
    f"export RUNNER_ALLOW_RUNASROOT=1 && cd {RUNNER_DIR} && ./config.sh"
    f" --url https://github.com/{REPO}"
    f" --token {reg_token}"
    f" --name {RUNNER_NAME}"
    f" --labels {RUNNER_LABELS}"
    f" --work /root/_work"
    f" --replace --unattended",
    shell=True, capture_output=True, text=True
)
if reg.returncode != 0:
    print("STDOUT:", reg.stdout[-800:])
    print("STDERR:", reg.stderr[-800:])
    raise RuntimeError("❌ config.sh échoué — voir logs ci-dessus")
print("✅ Runner enregistré")

print("\n🚀 Runner actif — déclenche le build depuis GitHub Actions")
print("─" * 50)
proc = subprocess.Popen(
    f"export RUNNER_ALLOW_RUNASROOT=1 && cd {RUNNER_DIR} && ./run.sh",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("\n🛑 Runner arrêté")